In [1]:
import pandas as pd
import requests
import json

# **API REQUETE GLOBALE** #

In [2]:
# CALL API
API_KEY = "haj00y9FzSQ1VZCTUjpmla8Q98xRnA6a"

URL = 'https://prim.iledefrance-mobilites.fr/marketplace/disruptions_bulk/disruptions/v2'

headers = {
    "Accept": "application/json",
    "apikey": API_KEY
}

response = requests.get(URL, headers=headers)

if response.status_code == 200:
    data = response.json()
    print(data.keys())  # Print the keys of the JSON response
else:
    print(f"Erreur {response.status_code}: {response.text}")

dict_keys(['disruptions', 'lines', 'lastUpdatedDate'])


## **Récupérer les données présente dna sles clés API** ##

In [3]:
# Accéder aux clés des perturbations
disruptions = data.get("disruptions", [])

In [4]:
disruptions

[{'id': '74316236-8627-11ef-aca5-0a58a9feac02',
  'applicationPeriods': [{'begin': '20241012T020900',
    'end': '20241013T233400'},
   {'begin': '20241019T120900', 'end': '20241020T230900'},
   {'begin': '20241207T020900', 'end': '20241208T234400'},
   {'begin': '20250118T020900', 'end': '20250119T234900'},
   {'begin': '20250329T020900', 'end': '20250330T235400'},
   {'begin': '20250405T030400', 'end': '20250406T234900'},
   {'begin': '20250412T030400', 'end': '20250413T234900'}],
  'lastUpdate': '20250401T120646',
  'cause': 'TRAVAUX',
  'severity': 'BLOQUANTE',
  'tags': ['Actualité'],
  'title': 'Travaux boulevard Carnot et Calmette',
  'message': '<p>En raison de travaux boulevard Carnot et Calmette à Mantes-la-Jolie, les arrêts suivants ne seront pas desservis :</p><p>Ligne D : (Plaisances) Carnot reporté à Louise Michel et Calmette reporté à Mantes-la-Ville Mairie</p><p>Ligne N : Calmette et Carnot reportés à Mantes-Station</p><p>Ligne A14 : Calmette reporté à Mantes-Station</p

In [5]:
# Accéder aux clés des arrêts concernés
lines = data.get("lines", [])

In [6]:
lines

[{'id': 'line:IDFM:C01423',
  'name': '1',
  'shortName': '1',
  'mode': 'Bus',
  'networkId': 'network:IDFM:6',
  'impactedObjects': [{'type': 'line',
    'id': 'line:IDFM:C01423',
    'name': '1',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']},
   {'type': 'stop_point',
    'id': 'stop_point:IDFM:16820',
    'name': 'Rond-Point des Sciences',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']},
   {'type': 'stop_point',
    'id': 'stop_point:IDFM:16819',
    'name': 'Rond-Point des Sciences',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']},
   {'type': 'stop_point',
    'id': 'stop_point:IDFM:19398',
    'name': 'Ampère',
    'disruptionIds': ['4884f0e6-00cd-11f0-af9a-0a58a9feac02']}]},
 {'id': 'line:IDFM:C02129',
  'name': '3',
  'shortName': '3',
  'mode': 'Bus',
  'networkId': 'network:IDFM:6',
  'impactedObjects': [{'type': 'line',
    'id': 'line:IDFM:C02129',
    'name': '3',
    'disruptionIds': ['5f95054a-d707-11ef-b937-0a58a9feac02',

## **Convertir les data de la clé diruption en Dataframe** ##

In [7]:
pd.reset_option('display.max_colwidth')

# Normalisation avec l'option errors='ignore' pour ignorer les clés manquantes
df_disruptions = pd.json_normalize(data['disruptions'], 
                                  record_path='applicationPeriods', 
                                  meta=['id', 'lastUpdate', 'cause', 'severity', 'tags', 'title', 'message', 'shortMessage'], 
                                  sep=',', errors='ignore')

# Affichage du résultat
df_disruptions.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage
0,20241012T020900,20241013T233400,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
1,20241019T120900,20241020T230900,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
2,20241207T020900,20241208T234400,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
3,20250118T020900,20250119T234900,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN
4,20250329T020900,20250330T235400,74316236-8627-11ef-aca5-0a58a9feac02,20250401T120646,TRAVAUX,BLOQUANTE,[Actualité],Travaux boulevard Carnot et Calmette,<p>En raison de travaux boulevard Carnot et Ca...,NaN


In [8]:
df_disruptions['severity'].value_counts()

severity
BLOQUANTE      3506
PERTURBEE      1638
INFORMATION     892
Name: count, dtype: int64

## **Convertir les data de la clé lines en Dataframe** ##

In [9]:
from collections import defaultdict

# 1. Créer un dictionnaire pour lier disruptionId aux arrêts concernés
disruption_to_stops = defaultdict(set)

# 2. Remplir ce dictionnaire et ajouter les informations 'name' et 'mode' à chaque disruptionId
disruption_line_info = {}  # Dictionnaire pour lier disruptionId aux informations de ligne (name, mode)

for line in lines:
    for obj in line.get('impactedObjects', []):
        if obj['type'] == 'stop_point':  # Filtrer uniquement les objets de type 'stop_point'
            stop_name = obj['name']
            for disruption_id in obj.get('disruptionIds', []):
                # Ajouter l'arrêt à la liste des arrêts pour ce disruptionId
                disruption_to_stops[disruption_id].add(stop_name)
                # Ajouter les informations de la ligne (name, mode) pour ce disruptionId
                if disruption_id not in disruption_line_info:
                    disruption_line_info[disruption_id] = {
                        'name': line['name'],
                        'mode': line['mode']
                    }

# 3. Construction du DataFrame avec les arrêts pour chaque disruptionId
df_lines = pd.DataFrame([
    {
        'disruptionId': disruption_id,
        'stop_points': sorted(list(stops)),
        'name': disruption_line_info[disruption_id]['name'],
        'mode': disruption_line_info[disruption_id]['mode'],
        
    }
    for disruption_id, stops in disruption_to_stops.items()
])

# Affichage du DataFrame final
df_lines.head()

,disruptionId,stop_points,name,mode
0,4884f0e6-00cd-11f0-af9a-0a58a9feac02,"[Ampère, Rond-Point des Sciences]",1,Bus
1,5f95054a-d707-11ef-b937-0a58a9feac02,"[Collège Maria Callas, Désiré Lefèvre, Le Chat...",3,Bus
2,570096d2-1399-11f0-bb7e-0a58a9feac02,"[Carrefour du 19 Mars 1962, Petit-Châtenay, Ru...",412,Bus
3,4f17c346-1399-11f0-bb7e-0a58a9feac02,"[Carrefour du 19 Mars 1962, Petit-Châtenay, Ru...",412,Bus
4,ff5f1896-d80a-11ef-b2cf-0a58a9feac02,"[Place Foch, Rue Andin]",5206 (ex P),Bus


## **Filtrer évènements en cours du dataset disruptions** ##

In [10]:
from datetime import datetime

def filtrer_evenements_en_cours(df, col_debut="begin", col_fin="end", now=None):
    """
    Filtre les événements en cours à partir d'un DataFrame avec colonnes 'start' et 'end' (format 'YYYYMMDDTHHMMSS').

    :param df: DataFrame contenant les événements
    :param col_debut: nom de la colonne de début (par défaut "start")
    :param col_fin: nom de la colonne de fin (par défaut "end")
    :param now: datetime personnalisé (utile pour les tests), sinon datetime.now()
    :return: DataFrame filtré avec les événements en cours
    """
    df = df.copy()
    
    # Conversion des dates
    df[col_debut] = pd.to_datetime(df[col_debut], format="%Y%m%dT%H%M%S", errors='coerce')
    df[col_fin] = pd.to_datetime(df[col_fin], format="%Y%m%dT%H%M%S", errors='coerce')

    # Date/heure actuelle
    now = now or pd.Timestamp.now()

    # Filtrage des événements en cours
    df_current = df[(df[col_debut] <= now) & (df[col_fin] >= now)]
    
    return df_current


In [11]:
df_current = filtrer_evenements_en_cours(df_disruptions)
df_current.head(2)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage
35,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","<p>Lignes 4414, 4438 et TàD Etréchy, du 6/01/2...",NaN
97,2025-04-08 00:00:00,2025-04-08 23:59:00,4e10e836-cea4-11ef-bc79-0a58a9feac02,20250228T154728,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,<p>En raison de travaux avenue Jean Jaurès à M...,NaN


In [12]:
def filtrer_evenements_previsionnels(df, col_debut="begin", col_fin="end", now=None):
    """
    Filtre les événements dont le début est dans les 2 heures à venir.

    :param df: DataFrame contenant les événements
    :param col_debut: nom de la colonne de début
    :param col_fin: nom de la colonne de fin
    :param now: datetime personnalisé (utile pour les tests), sinon datetime.now()
    :return: DataFrame filtré avec les événements débutant dans les 2 prochaines heures
    """
    df = df.copy()
    
    # Conversion des dates
    df[col_debut] = pd.to_datetime(df[col_debut], format="%Y%m%dT%H%M%S", errors='coerce')
    df[col_fin] = pd.to_datetime(df[col_fin], format="%Y%m%dT%H%M%S", errors='coerce')

    # Date/heure actuelle
    now = now or pd.Timestamp.now()
    dans_2h = now + pd.Timedelta(hours=7)

    # Filtrage des événements qui commenceront dans les 2 prochaines heures
    df_preview = df[(df[col_debut] >= now) & (df[col_debut] <= dans_2h)]

    return df_preview


In [13]:
df_preview = filtrer_evenements_previsionnels(df_disruptions)
df_preview.head(2)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage
28,2025-04-08 22:00:00,2025-04-09 04:30:00,e0812420-ad56-11ef-b233-0a58a9feac02,20241128T080331,TRAVAUX,BLOQUANTE,[Actualité],Métro 14 : Travaux - Trafic interrompu,"<p>Du 3 mars au 15 avril, les lundi et mardi à...",NaN
653,2025-04-08 22:00:00,2025-04-09 04:30:00,9b41a11c-f90c-11ef-af9a-0a58a9feac02,20250305T105203,TRAVAUX,BLOQUANTE,[Actualité],Ligne 480 (Seine Grand Orly) - Arrêt Orly 4 no...,<p><strong><u>Ligne 480 (Seine Grand Orly) - A...,NaN


## **Merged dataset disruptions et lines sur les clés communes disruptionId et id** ##

In [14]:
# On renomme "disruptionId" en "id" dans df_lines pour pouvoir faire la jointure facilement
df_lines_renamed = df_lines.rename(columns={'disruptionId': 'id'})

# Merge (left join) : on garde tout df_en_cours, et on ajoute les colonnes de df_lines si id commun
df_current_merged = df_current.merge(df_lines_renamed, on='id', how='left')

In [15]:
# Merge (left join) : on garde tout df_en_cours, et on ajoute les colonnes de df_lines si id commun
df_preview_merged = df_preview.merge(df_lines_renamed, on='id', how='left')

## **PREPROCESSING** ##

In [16]:
from bs4 import BeautifulSoup
# Supprimer valeurs NAN et converttir en str
df_current_merged['message'] = df_current_merged['message'].astype(str).fillna('')

# Appliquer la suppression des balises HTML à chaque message de la colonne 'message'
df_current_merged['message'] = df_current_merged['message'].apply(
    lambda msg: BeautifulSoup(msg, "html.parser").get_text()
)

In [17]:
# Supprimer valeurs NAN et converttir en str
df_preview_merged['message'] = df_preview_merged['message'].astype(str).fillna('')

# Appliquer la suppression des balises HTML à chaque message de la colonne 'message'
df_preview_merged['message'] = df_preview_merged['message'].apply(
    lambda msg: BeautifulSoup(msg, "html.parser").get_text()
)

In [18]:
df_current_merged.head(2)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode
0,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","Lignes 4414, 4438 et TàD Etréchy, du 6/01/2025...",NaN,[Grande Rue],4414,Bus
1,2025-04-08 00:00:00,2025-04-08 23:59:00,4e10e836-cea4-11ef-bc79-0a58a9feac02,20250228T154728,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,"[Louise Michel, Mantes Station, Mantes-la-Vill...",88C,Bus


In [19]:
df_preview_merged

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode
0,2025-04-08 22:00:00,2025-04-09 04:30:00,e0812420-ad56-11ef-b233-0a58a9feac02,20241128T080331,TRAVAUX,BLOQUANTE,[Actualité],Métro 14 : Travaux - Trafic interrompu,"Du 3 mars au 15 avril, les lundi et mardi à pa...",NaN,"[Bercy, Bibliothèque François Mitterrand, Chât...",14,Metro
1,2025-04-08 22:00:00,2025-04-09 04:30:00,9b41a11c-f90c-11ef-af9a-0a58a9feac02,20250305T105203,TRAVAUX,BLOQUANTE,[Actualité],Ligne 480 (Seine Grand Orly) - Arrêt Orly 4 no...,Ligne 480 (Seine Grand Orly) - Arrêt non desse...,NaN,[Aéroport Orly 4],480,Bus
2,2025-04-08 22:00:00,2025-04-09 04:30:00,640bcc8c-043c-11f0-9201-0a58a9feac02,20250318T220307,TRAVAUX,BLOQUANTE,[Actualité],Métro 10 : Travaux - Trafic interrompu,"Jusqu'au 29 avril (sauf le 9 avril), du lundi ...",Trafic interrompu,NaN,NaN,NaN
3,2025-04-08 22:45:00,2025-04-09 02:45:00,e5f599c4-0632-11f0-9201-0a58a9feac02,20250321T100012,TRAVAUX,BLOQUANTE,[Actualité],Ligne B : Châtelet - CDG2/Mitry du 31/03 au 25/04,"Période : du lundi au vendredi, à partir de 22...",Trafic interrompu planifié,"[Antony, Arcueil - Cachan, Aulnay-sous-Bois, A...",B,RapidTransit
4,2025-04-08 22:00:00,2025-04-09 04:30:00,4e42f940-0f34-11f0-8f48-0a58a9feac02,20250401T220258,TRAVAUX,BLOQUANTE,[Actualité],Métro 14 : Travaux - Trafic interrompu,"Jusqu'au 29 avril, les lundi et mardi à partir...",Trafic interrompu,"[Bercy, Bibliothèque François Mitterrand, Chât...",14,Metro
5,2025-04-08 22:30:00,2025-04-08 23:30:00,ff978aca-13b3-11f0-b7af-0a58a9feac02,20250407T152710,TRAVAUX,BLOQUANTE,[Actualité],Ligne D : Paris Lyon<>Villeneuve : du 31/03 au...,Période : à partir de 22h30.Date : lundi 31 ma...,Arrêts non desservis planifiés,"[Ballancourt, Boigneville, Boussy-Saint-Antoin...",D,RapidTransit
6,2025-04-08 21:00:00,2025-04-09 02:40:00,e55e5334-141f-11f0-8f48-0a58a9feac02,20250408T041927,TRAVAUX,BLOQUANTE,NaN,Bus 112 : Travaux - Arrêt(s) non desservi(s),La ligne 112 sera déviée : les arrêts situés e...,Arrêt(s) non desservi(s),"[Carrefour de Beauté, Cartoucherie, Champ de M...",112,Bus
7,2025-04-08 21:00:00,2025-04-09 02:40:00,ec9e5a2c-141f-11f0-88c2-0a58a9feac02,20250408T041941,TRAVAUX,BLOQUANTE,NaN,Bus 112 : Travaux - Arrêt(s) non desservi(s),La ligne 112 sera déviée : les arrêts situés e...,Arrêt(s) non desservi(s),"[Carrefour de Beauté, Cartoucherie, Champ de M...",112,Bus
8,2025-04-08 20:00:00,2025-04-09 03:00:00,1a79691c-1422-11f0-bb7e-0a58a9feac02,20250408T043515,PERTURBATION,BLOQUANTE,NaN,Bus 235 : Incident sur la voie publique - Arrê...,La ligne 235 sera déviée : les arrêts situés e...,Arrêt(s) non desservi(s),"[Cité du Luth-Verlaine, Deslandes, Le Luth, Le...",235,Bus
9,2025-04-08 20:00:00,2025-04-09 03:00:00,23bf9334-1422-11f0-88c2-0a58a9feac02,20250408T043531,PERTURBATION,BLOQUANTE,NaN,Bus 235 : Incident sur la voie publique - Arrê...,La ligne 235 sera déviée : les arrêts situés e...,Arrêt(s) non desservi(s),"[Cité du Luth-Verlaine, Deslandes, Le Luth, Le...",235,Bus


In [20]:
# Récupérer les ID des perturbations du précédent appel API
# Par exemple, ces ID pourraient être stockés dans un fichier ou une base de données.
previous_disruptions = set(df_current_merged['id'].tolist())

# Simulons un nouvel appel API en modifiant df_merged pour représenter le nouvel état des perturbations
# Ce dataframe (df_merged) serait mis à jour à chaque appel API avec de nouvelles données.
new_disruptions = set(df_current_merged['id'].tolist())  # Ici on reprend la même liste, mais dans un vrai cas, ce serait mis à jour.

# Ajouter la colonne 'status' en fonction des conditions
df_current_merged['status'] = df_current_merged['id'].apply(
    lambda x: 'new' if x not in previous_disruptions else ('finished' if x not in new_disruptions else 'now')
)

# Afficher le DataFrame avec la nouvelle colonne 'status'
df_current_merged.head()

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status
0,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","Lignes 4414, 4438 et TàD Etréchy, du 6/01/2025...",NaN,[Grande Rue],4414,Bus,now
1,2025-04-08 00:00:00,2025-04-08 23:59:00,4e10e836-cea4-11ef-bc79-0a58a9feac02,20250228T154728,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,"[Louise Michel, Mantes Station, Mantes-la-Vill...",88C,Bus,now
2,2025-04-08 00:00:00,2025-04-08 23:59:00,0bbdedb6-cea5-11ef-b1c7-0a58a9feac02,20250218T162628,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,NaN,NaN,NaN,now
3,2025-01-13 00:00:00,2025-12-31 23:59:00,30376efa-cea6-11ef-8d37-0a58a9feac02,20250109T172424,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,[Poste],21,Bus,now
4,2025-01-13 00:00:00,2025-06-30 23:59:00,426b148c-ceab-11ef-8d37-0a58a9feac02,20250313T203153,TRAVAUX,BLOQUANTE,NaN,🚧 5121 5152 - Travaux : Arrêt Golf National da...,🚧 #Perturbations #Ligne5121 #Ligne5152📅 à part...,NaN,[Golf National],5121,Bus,now


## **Ouvrir le précédent appel API s'il existe** ##

In [21]:
# Charger le fichier CSV de la version précédente
historic_api_file = "df_previous_merged.csv"

try:
    df_previous = pd.read_csv(historic_api_file)
    print("Fichier précédent chargé depuis CSV.")
except FileNotFoundError:
    df_previous = pd.DataFrame(columns=["id", "name", "mode", "begin", "end", "severity", "tags", "title", "message", "status"])
    print("Aucun fichier précédent trouvé, c’est le premier appel ?")

Fichier précédent chargé depuis CSV.


## **Remplacer l'ancien appel API par l'actuel qui deviendra lui même l'ancien appel API** ##

In [22]:
df_current_merged.to_csv(f"df_previous_merged.csv", index=False)

In [23]:
df_current = df_current_merged

## **Mise à jour du statut des disruptions : nouvelles, en cours, terminées** ##


In [24]:
# Ajouter la colonne 'status' pour les disruptions actuelles
df_current["status"] = df_current["id"].apply(
    lambda x: "new" if x not in df_previous["id"].values else "now"
)

# Identifier les disruptions terminées ("finished")
finished_ids = set(df_previous["id"]) - set(df_current["id"])
df_finished = df_previous[df_previous["id"].isin(finished_ids)].copy()
df_finished["status"] = "finished"  # Ajouter le statut "finished"

## **Concaténer les résultats** ##

In [25]:
# Fusionner les disruptions actuelles et terminées
df_current_histo = pd.concat([df_current, df_finished], ignore_index=True)

In [26]:
df_current_histo.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status
0,2025-01-06 01:00:00,2025-06-06 23:00:00,4a2946e2-beae-11ef-b1c7-0a58a9feac02,20241220T094303,TRAVAUX,BLOQUANTE,NaN,"4414, 4438 et TàD Etréchy, Travaux Chauffour-l...","Lignes 4414, 4438 et TàD Etréchy, du 6/01/2025...",NaN,[Grande Rue],4414,Bus,now
1,2025-04-08 00:00:00,2025-04-08 23:59:00,4e10e836-cea4-11ef-bc79-0a58a9feac02,20250228T154728,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,"[Louise Michel, Mantes Station, Mantes-la-Vill...",88C,Bus,now
2,2025-04-08 00:00:00,2025-04-08 23:59:00,0bbdedb6-cea5-11ef-b1c7-0a58a9feac02,20250218T162628,TRAVAUX,BLOQUANTE,[Actualité],Fermeture avenue Jean Jaurès à Mantes-la-Ville...,En raison de travaux avenue Jean Jaurès à Mant...,NaN,NaN,NaN,NaN,now


In [27]:
df_current_histo['mode'].value_counts()

mode
Bus             280
RapidTransit      6
LocalTrain        3
Tramway           1
Metro             1
Name: count, dtype: int64

## **Filtrer les disruptions concernant le réseau férré** ##

In [28]:
df_current_filtered = df_current_histo[df_current_histo['mode'].isin(['RapidTransit', 'LocalTrain', 'Tramway', 'Metro'])]

In [29]:
df_preview_filtered = df_preview_merged[df_preview_merged['mode'].isin(['RapidTransit', 'LocalTrain', 'Tramway', 'Metro'])]

In [30]:
df_preview_filtered.shape

(12, 13)

In [31]:
df_preview_filtered.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode
0,2025-04-08 22:00:00,2025-04-09 04:30:00,e0812420-ad56-11ef-b233-0a58a9feac02,20241128T080331,TRAVAUX,BLOQUANTE,[Actualité],Métro 14 : Travaux - Trafic interrompu,"Du 3 mars au 15 avril, les lundi et mardi à pa...",NaN,"[Bercy, Bibliothèque François Mitterrand, Chât...",14,Metro
3,2025-04-08 22:45:00,2025-04-09 02:45:00,e5f599c4-0632-11f0-9201-0a58a9feac02,20250321T100012,TRAVAUX,BLOQUANTE,[Actualité],Ligne B : Châtelet - CDG2/Mitry du 31/03 au 25/04,"Période : du lundi au vendredi, à partir de 22...",Trafic interrompu planifié,"[Antony, Arcueil - Cachan, Aulnay-sous-Bois, A...",B,RapidTransit
4,2025-04-08 22:00:00,2025-04-09 04:30:00,4e42f940-0f34-11f0-8f48-0a58a9feac02,20250401T220258,TRAVAUX,BLOQUANTE,[Actualité],Métro 14 : Travaux - Trafic interrompu,"Jusqu'au 29 avril, les lundi et mardi à partir...",Trafic interrompu,"[Bercy, Bibliothèque François Mitterrand, Chât...",14,Metro


In [32]:
df_current_filtered.shape

(11, 14)

In [33]:
df_current_filtered['status'].value_counts()

status
finished    5
now         4
new         2
Name: count, dtype: int64

In [34]:
df_current_filtered.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status
60,2025-03-27 18:42:00,2025-04-22 04:30:00,26847bb8-0b33-11f0-b7af-0a58a9feac02,20250327T184436,TRAVAUX,BLOQUANTE,[Actualité],Tramway T1 : Travaux - Trafic interrompu,"Jusqu'au lundi 21 avril 2025, le trafic est in...",Trafic interrompu,"[Auguste Delaune, Bobigny - Pablo Picasso, Esc...",T1,Tramway,now
316,2025-04-08 08:15:00,2025-04-08 16:20:00,99c8f304-f2c1-11ef-96bd-0a58a9feac02,20250224T161119,TRAVAUX,PERTURBEE,[Actualité],Ligne E : Nanterre- Chelles 7 - 11/04,Période : de 8h15 à 16h15Dates : du lundi 7 au...,trafic ralenti,"[Bondy, Chelles - Gournay, Gagny, Haussmann Sa...",E,RapidTransit,now
367,2025-04-08 09:00:00,2025-04-08 16:00:00,97f26936-0ed5-11f0-ac05-0a58a9feac02,20250401T104458,TRAVAUX,PERTURBEE,[Actualité],Ligne D Goussainville Melun 03 mars au 22 août...,Période : de 09h00 à 16h00.Dates : du lundi 3 ...,trafic ralenti,"[Boussy-Saint-Antoine, Brunoy, Châtelet les Ha...",D,RapidTransit,now


In [35]:
# CALL API
API_KEY = "haj00y9FzSQ1VZCTUjpmla8Q98xRnA6a"

URL = 'https://prim.iledefrance-mobilites.fr/marketplace/v2/navitia/line_reports/line_reports'

headers = {
    "Accept": "application/json",
    "apikey": API_KEY
}

response = requests.get(URL, headers=headers)

if response.status_code == 200:
    data_2 = response.json()
    print(data.keys())
else:
    print(f"Erreur {response.status_code}: {response.text}")

dict_keys(['disruptions', 'lines', 'lastUpdatedDate'])


In [36]:
disruptions_pag_v2 = data_2.get("pagination", [])

In [37]:
disruptions_pag_v2

{'total_result': 863,
 'start_page': 0,
 'items_per_page': 25,
 'items_on_page': 25}

In [38]:
# Accéder aux clés des perturbations
disruptions_v2 = data_2.get("disruptions", [])

In [39]:
disruptions_v2

[{'id': '5f95054a-d707-11ef-b937-0a58a9feac02',
  'disruption_id': 'b0e9ae60-d3f0-11ef-aa1a-0a58a9feac02',
  'impact_id': '5f95054a-d707-11ef-b937-0a58a9feac02',
  'application_periods': [{'begin': '20250120T040000',
    'end': '20250530T230000'}],
  'status': 'active',
  'updated_at': '20250226T152150',
  'cause': 'travaux',
  'category': 'Incidents',
  'severity': {'name': 'Bloquante passante',
   'effect': 'MODIFIED_SERVICE',
   'color': '#FF0000',
   'priority': 0},
  'messages': [{'text': '<p>En raison de travaux, avenue de Chantereine, une déviation doit être mise en place.&nbsp;<br>Du lundi 20 janvier au vendredi 30 mai 2025.<br>- En direction de la Gare de Chelles : les arrêts Collège Maria Callas, Le Chat ZI, Onze Arpents, Désiré Lefèvre, Les Clos, Levasseur ne seront pas desservis.&nbsp;<br>Merci de vous reporter aux arrêts La Régale (poteau ligne 5 direction Gare de Lagny-Thorigny), ou Chantereine.&nbsp;<br>- En direction de la Gare de Villeparisis / Eglise de Courtry : les 

In [40]:
import requests
import pandas as pd

# URL de base de l'API
base_url = "https://prim.iledefrance-mobilites.fr/marketplace/v2/navitia/line_reports/line_reports"
headers = {
    "apikey": API_KEY  # Remplace avec ta vraie clé API
}

# Pagination
start_page = 0
items_per_page = 25
all_rows = []

while True:
    params = {
        "start_page": start_page,
        "count": items_per_page
    }
    
    response = requests.get(base_url, headers=headers, params=params)
    
    if response.status_code != 200:
        print(f"Erreur {response.status_code} à la page {start_page}")
        break

    json_data = response.json()
    disruptions = json_data.get("disruptions", [])

    if not disruptions:
        break  # Plus de disruptions à parcourir

    # Traitement de chaque disruption
    for disruption in disruptions:
        html_message = next(
            (msg['text'] for msg in disruption.get('messages', [])
             if msg.get('channel', {}).get('content_type') == 'text/html'),
            None
        )
        
        row = {
            'priority': disruption['severity']['priority'],
            'effect': disruption['severity']['effect'],
            'severity_text': disruption['severity']['name'],
            #'begin': disruption['application_periods'][0]['begin'],
            #'end': disruption['application_periods'][0]['end'],
            'id': disruption['id'],
            'text': html_message
        }
        all_rows.append(row)

    print(f"Page {start_page} traitée avec {len(disruptions)} disruptions.")
    
    # Vérification fin de pagination
    pagination = json_data.get('pagination', {})
    total_result = pagination.get('total_result', 0)
    if (start_page + 1) * items_per_page >= total_result:
        break

    start_page += 1

# Construction du DataFrame
df_v2 = pd.DataFrame(all_rows)


Page 0 traitée avec 15 disruptions.
Page 1 traitée avec 26 disruptions.
Page 2 traitée avec 18 disruptions.
Page 3 traitée avec 22 disruptions.
Page 4 traitée avec 9 disruptions.
Page 5 traitée avec 31 disruptions.
Page 6 traitée avec 16 disruptions.
Page 7 traitée avec 25 disruptions.
Page 8 traitée avec 92 disruptions.
Page 9 traitée avec 22 disruptions.
Page 10 traitée avec 21 disruptions.
Page 11 traitée avec 7 disruptions.
Page 12 traitée avec 13 disruptions.
Page 13 traitée avec 17 disruptions.
Page 14 traitée avec 24 disruptions.
Page 15 traitée avec 57 disruptions.
Page 16 traitée avec 51 disruptions.
Page 17 traitée avec 70 disruptions.
Page 18 traitée avec 62 disruptions.
Page 19 traitée avec 60 disruptions.
Page 20 traitée avec 54 disruptions.
Page 21 traitée avec 70 disruptions.
Page 22 traitée avec 58 disruptions.
Page 23 traitée avec 47 disruptions.
Page 24 traitée avec 68 disruptions.
Page 25 traitée avec 141 disruptions.
Page 26 traitée avec 199 disruptions.
Page 27 tra

In [41]:
df_v2.head(3)

,priority,effect,severity_text,id,text
0,0,MODIFIED_SERVICE,Bloquante passante,5f95054a-d707-11ef-b937-0a58a9feac02,"<p>En raison de travaux, avenue de Chantereine..."
1,0,MODIFIED_SERVICE,Bloquante passante,4884f0e6-00cd-11f0-af9a-0a58a9feac02,<p>En direction de Chelles : les arrêts Ampère...
2,30,SIGNIFICANT_DELAYS,perturbée,beac34e6-297d-11ef-9244-0a58a9feac02,SURESNES / LONGCHAMP Panne de l'ascenseur situ...


provisoire en dessous

In [42]:
"""# Conversion des dates
df_current_filtered.loc[:, 'begin'] = pd.to_datetime(df_current_filtered['begin'], format="%Y%m%dT%H%M%S", errors='coerce')
df_current_filtered.loc[:, 'end'] = pd.to_datetime(df_current_filtered['end'], format="%Y%m%dT%H%M%S", errors='coerce')

# Conversion des dates
df_v2['begin'] = pd.to_datetime(df_v2['begin'], format="%Y%m%dT%H%M%S", errors='coerce')
df_v2['end'] = pd.to_datetime(df_v2['end'], format="%Y%m%dT%H%M%S", errors='coerce')"""

'# Conversion des dates\ndf_current_filtered.loc[:, \'begin\'] = pd.to_datetime(df_current_filtered[\'begin\'], format="%Y%m%dT%H%M%S", errors=\'coerce\')\ndf_current_filtered.loc[:, \'end\'] = pd.to_datetime(df_current_filtered[\'end\'], format="%Y%m%dT%H%M%S", errors=\'coerce\')\n\n# Conversion des dates\ndf_v2[\'begin\'] = pd.to_datetime(df_v2[\'begin\'], format="%Y%m%dT%H%M%S", errors=\'coerce\')\ndf_v2[\'end\'] = pd.to_datetime(df_v2[\'end\'], format="%Y%m%dT%H%M%S", errors=\'coerce\')'

In [43]:
df_current_v2_merged = pd.merge(df_current_filtered, df_v2, on=['id'], how='left')
#df_current_v2_merged = pd.merge(df_current_filtered, df_v2, on=['id', 'begin', 'end'], how='left')

In [44]:
df_current_v2_merged.shape

(11, 18)

In [45]:
df_current_v2_merged.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status,priority,effect,severity_text,text
0,2025-03-27 18:42:00,2025-04-22 04:30:00,26847bb8-0b33-11f0-b7af-0a58a9feac02,20250327T184436,TRAVAUX,BLOQUANTE,[Actualité],Tramway T1 : Travaux - Trafic interrompu,"Jusqu'au lundi 21 avril 2025, le trafic est in...",Trafic interrompu,"[Auguste Delaune, Bobigny - Pablo Picasso, Esc...",T1,Tramway,now,0.0,NO_SERVICE,bloquante,"<p>Jusqu'au lundi 21 avril 2025, le trafic est..."
1,2025-04-08 08:15:00,2025-04-08 16:20:00,99c8f304-f2c1-11ef-96bd-0a58a9feac02,20250224T161119,TRAVAUX,PERTURBEE,[Actualité],Ligne E : Nanterre- Chelles 7 - 11/04,Période : de 8h15 à 16h15Dates : du lundi 7 au...,trafic ralenti,"[Bondy, Chelles - Gournay, Gagny, Haussmann Sa...",E,RapidTransit,now,30.0,SIGNIFICANT_DELAYS,perturbée,<p>P&#233;riode : de 8h15 &#224; 16h15<br><br>...
2,2025-04-08 09:00:00,2025-04-08 16:00:00,97f26936-0ed5-11f0-ac05-0a58a9feac02,20250401T104458,TRAVAUX,PERTURBEE,[Actualité],Ligne D Goussainville Melun 03 mars au 22 août...,Période : de 09h00 à 16h00.Dates : du lundi 3 ...,trafic ralenti,"[Boussy-Saint-Antoine, Brunoy, Châtelet les Ha...",D,RapidTransit,now,30.0,SIGNIFICANT_DELAYS,perturbée,<p>P&#233;riode : de 09h00 &#224; 16h00.<br><b...


In [46]:
df_current_v2_merged['text'][0]

"<p>Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen en raison de travaux. .<br>Bus de remplacement.<br><a href='http://www.ratp.fr'>Plus d'informations sur le site ratp.fr</a></p>"

# recuperer le nombre de validations de voyageurs 

In [65]:
n_validation = pd.read_csv("../data/nb_vald_par_arret_jour.csv", sep=";", encoding="utf-8")
n_validation.iloc[177:183].head(10)
# Filtrer les données pour une date spécifique
aujourd_hui = datetime.now()
#aujourd_hui = datetime(2023, 10, 1)  # Exemple de date spécifique
yearFormat = aujourd_hui.strftime('%Y-%m-%d') 
lastYearToday = (aujourd_hui.replace(year=aujourd_hui.year - 1)).strftime('%Y-%m-%d')
# Afficher les résultats
print(f"la date aujourd_hui {aujourd_hui} :")
print(f"la date yearFormat {yearFormat} :")
print(f"la date lastYear {lastYearToday} :")

la date aujourd_hui 2025-04-08 16:22:38.875086 :
la date yearFormat 2025-04-08 :
la date lastYear 2024-04-08 :


In [66]:
# Convertir les noms des arrêts en minuscules
n_validation['libelle_arret'] = n_validation['libelle_arret'].str.lower()
numb = 0
stations_valid_day = []
# Filtrer les données pour la date spécifique
filtered_validation = n_validation[n_validation['jour'] == lastYearToday]

# Vérifier les arrêts dans df_current_v2_merged
for stop_points in df_current_v2_merged['stop_points']:
    for stop in stop_points:
        if stop.lower() in filtered_validation['libelle_arret'].values:
            # Récupérer le nombre de validations correspondant
            nb_vald = filtered_validation.loc[filtered_validation['libelle_arret'] == stop.lower(), 'nb_vald'].values
            if len(nb_vald) > 0:
                stations_valid_day.append(f"{stop} {nb_vald[0]}")
                numb += 1

print(numb)
print(stations_valid_day)

21
['Magenta 6075', 'Pantin 8862', 'Rosa Parks 14798', 'Brunoy 5782', 'Gare de Lyon 115382', 'Gare du Nord 105502', 'Goussainville 5443', 'Yerres 4483', 'Brunoy 5782', 'Gare de Lyon 115382', 'Gare du Nord 105502', 'Goussainville 5443', 'Yerres 4483', 'Bougival 686', 'Courbevoie 2967', 'Louveciennes 694', 'Pont Cardinet 7', 'Puteaux 2787', 'Vaucresson 1538', 'Chartrettes 24', 'Montereau 3058']


In [ ]:
from datetime import datetime

# Convertir les colonnes 'begin' et 'end' en datetime si ce n'est pas déjà fait
df_current_v2_merged['begin'] = pd.to_datetime(df_current_v2_merged['begin'], format='%Y%m%dT%H%M%S', errors='coerce')
df_current_v2_merged['end'] = pd.to_datetime(df_current_v2_merged['end'], format='%Y%m%dT%H%M%S', errors='coerce')

# Ajouter une colonne 'total_valid_imp' initialisée à 0
df_current_v2_merged['total_valid_imp'] = 0

# Filtrer les données pour la date spécifique
filtered_validation = n_validation[n_validation['jour'] == lastYearToday]

# Parcourir chaque ligne de df_current_v2_merged
for index, row in df_current_v2_merged.iterrows():
    stop_points = row['stop_points']  # Liste des arrêts impactés
    begin = row['begin']
    end = row['end']
    
    # Vérifier que begin et end sont valides
    if pd.notnull(begin) and pd.notnull(end):
        # Calculer la durée en heures entre begin et end
        duration_hours = (end - begin).total_seconds() / 3600
        
        # Filtrer les validations pour les arrêts impactés
        total_validations = 0
        for stop in stop_points:
            stop_lower = stop.lower()  # Convertir en minuscules pour correspondance
            if stop_lower in filtered_validation['libelle_arret'].str.lower().values:
                # Récupérer le nombre de validations correspondant
                nb_vald = filtered_validation.loc[filtered_validation['libelle_arret'].str.lower() == stop_lower, 'nb_vald'].values
                if len(nb_vald) > 0:
                    total_validations += nb_vald[0]
        
        # Réduire le total des validations en fonction de la durée
        # Supposons que les validations sont réparties uniformément sur 24 heures
        impacted_validations = total_validations * (duration_hours / 24)
        
        # Ajouter le résultat dans la colonne 'total_valid_imp'
        df_current_v2_merged.at[index, 'total_valid_imp'] = round(impacted_validations)

# Afficher un aperçu du DataFrame mis à jour
print(df_current_v2_merged[['id' ,'stop_points', 'begin', 'end', 'total_valid_imp']].head())

                                     id  \
0  26847bb8-0b33-11f0-b7af-0a58a9feac02   
1  99c8f304-f2c1-11ef-96bd-0a58a9feac02   
2  97f26936-0ed5-11f0-ac05-0a58a9feac02   
3  6bd130e6-10c7-11f0-8f48-0a58a9feac02   
4  cb00c832-1479-11f0-88c2-0a58a9feac02   

                                         stop_points               begin  \
0  [Auguste Delaune, Bobigny - Pablo Picasso, Esc... 2025-03-27 18:42:00   
1  [Bondy, Chelles - Gournay, Gagny, Haussmann Sa... 2025-04-08 08:15:00   
2  [Boussy-Saint-Antoine, Brunoy, Châtelet les Ha... 2025-04-08 09:00:00   
3  [Boussy-Saint-Antoine, Brunoy, Châtelet les Ha... 2025-04-08 09:00:00   
4  [Asnières-sur-Seine, Bougival, Bécon les Bruyè... 2025-04-08 14:42:14   

                  end  total_valid_imp  
0 2025-04-22 04:30:00                0  
1 2025-04-08 16:20:00            10015  
2 2025-04-08 16:00:00            69006  
3 2025-04-08 16:00:00            69006  
4 2025-04-08 16:00:00              469  


In [68]:
df_current_v2_merged.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status,priority,effect,severity_text,text,total_valid_imp
0,2025-03-27 18:42:00,2025-04-22 04:30:00,26847bb8-0b33-11f0-b7af-0a58a9feac02,20250327T184436,TRAVAUX,BLOQUANTE,[Actualité],Tramway T1 : Travaux - Trafic interrompu,"Jusqu'au lundi 21 avril 2025, le trafic est in...",Trafic interrompu,"[Auguste Delaune, Bobigny - Pablo Picasso, Esc...",T1,Tramway,now,0.0,NO_SERVICE,bloquante,"<p>Jusqu'au lundi 21 avril 2025, le trafic est...",0
1,2025-04-08 08:15:00,2025-04-08 16:20:00,99c8f304-f2c1-11ef-96bd-0a58a9feac02,20250224T161119,TRAVAUX,PERTURBEE,[Actualité],Ligne E : Nanterre- Chelles 7 - 11/04,Période : de 8h15 à 16h15Dates : du lundi 7 au...,trafic ralenti,"[Bondy, Chelles - Gournay, Gagny, Haussmann Sa...",E,RapidTransit,now,30.0,SIGNIFICANT_DELAYS,perturbée,<p>P&#233;riode : de 8h15 &#224; 16h15<br><br>...,10015
2,2025-04-08 09:00:00,2025-04-08 16:00:00,97f26936-0ed5-11f0-ac05-0a58a9feac02,20250401T104458,TRAVAUX,PERTURBEE,[Actualité],Ligne D Goussainville Melun 03 mars au 22 août...,Période : de 09h00 à 16h00.Dates : du lundi 3 ...,trafic ralenti,"[Boussy-Saint-Antoine, Brunoy, Châtelet les Ha...",D,RapidTransit,now,30.0,SIGNIFICANT_DELAYS,perturbée,<p>P&#233;riode : de 09h00 &#224; 16h00.<br><b...,69006


In [69]:
df_current_v2_merged.to_csv('df_current_final.csv', index = False)
df_preview_filtered.to_csv('df_preview_final.csv', index = False)

In [70]:
df_current_final = pd.read_csv('df_current_final.csv')
df_current_final.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode,status,priority,effect,severity_text,text,total_valid_imp
0,2025-03-27 18:42:00,2025-04-22 04:30:00,26847bb8-0b33-11f0-b7af-0a58a9feac02,20250327T184436,TRAVAUX,BLOQUANTE,['Actualité'],Tramway T1 : Travaux - Trafic interrompu,"Jusqu'au lundi 21 avril 2025, le trafic est in...",Trafic interrompu,"['Auguste Delaune', 'Bobigny - Pablo Picasso',...",T1,Tramway,now,0.0,NO_SERVICE,bloquante,"<p>Jusqu'au lundi 21 avril 2025, le trafic est...",0
1,2025-04-08 08:15:00,2025-04-08 16:20:00,99c8f304-f2c1-11ef-96bd-0a58a9feac02,20250224T161119,TRAVAUX,PERTURBEE,['Actualité'],Ligne E : Nanterre- Chelles 7 - 11/04,Période : de 8h15 à 16h15Dates : du lundi 7 au...,trafic ralenti,"['Bondy', 'Chelles - Gournay', 'Gagny', 'Hauss...",E,RapidTransit,now,30.0,SIGNIFICANT_DELAYS,perturbée,<p>P&#233;riode : de 8h15 &#224; 16h15<br><br>...,10015
2,2025-04-08 09:00:00,2025-04-08 16:00:00,97f26936-0ed5-11f0-ac05-0a58a9feac02,20250401T104458,TRAVAUX,PERTURBEE,['Actualité'],Ligne D Goussainville Melun 03 mars au 22 août...,Période : de 09h00 à 16h00.Dates : du lundi 3 ...,trafic ralenti,"['Boussy-Saint-Antoine', 'Brunoy', 'Châtelet l...",D,RapidTransit,now,30.0,SIGNIFICANT_DELAYS,perturbée,<p>P&#233;riode : de 09h00 &#224; 16h00.<br><b...,69006


In [71]:
df_preview_final = pd.read_csv('df_preview_final.csv')
df_preview_final.head(3)

,begin,end,id,lastUpdate,cause,severity,tags,title,message,shortMessage,stop_points,name,mode
0,2025-04-08 22:00:00,2025-04-09 04:30:00,e0812420-ad56-11ef-b233-0a58a9feac02,20241128T080331,TRAVAUX,BLOQUANTE,['Actualité'],Métro 14 : Travaux - Trafic interrompu,"Du 3 mars au 15 avril, les lundi et mardi à pa...",NaN,"['Bercy', 'Bibliothèque François Mitterrand', ...",14,Metro
1,2025-04-08 22:45:00,2025-04-09 02:45:00,e5f599c4-0632-11f0-9201-0a58a9feac02,20250321T100012,TRAVAUX,BLOQUANTE,['Actualité'],Ligne B : Châtelet - CDG2/Mitry du 31/03 au 25/04,"Période : du lundi au vendredi, à partir de 22...",Trafic interrompu planifié,"['Antony', 'Arcueil - Cachan', 'Aulnay-sous-Bo...",B,RapidTransit
2,2025-04-08 22:00:00,2025-04-09 04:30:00,4e42f940-0f34-11f0-8f48-0a58a9feac02,20250401T220258,TRAVAUX,BLOQUANTE,['Actualité'],Métro 14 : Travaux - Trafic interrompu,"Jusqu'au 29 avril, les lundi et mardi à partir...",Trafic interrompu,"['Bercy', 'Bibliothèque François Mitterrand', ...",14,Metro


In [61]:
#print(df_preview_final.isnull().sum())

In [62]:
#print(df_current_final.isnull().sum())

In [76]:
import pandas as pd
import ast
from bs4 import BeautifulSoup

def clean_and_describe_csv(file_path, title):
    # Lire le fichier CSV
    df = pd.read_csv(file_path)
    df = df.fillna(" ")  # Remplacer les NaN par une chaîne vide
    # Nettoyer les colonnes
    df['tags'] = df['tags'].apply(ast.literal_eval)
    df['stop_points'] = df['stop_points'].apply(ast.literal_eval)

    # Nettoyer la colonne 'text' si elle existe
    if 'text' in df.columns:
        df['text'] = df['text'].apply(lambda x: BeautifulSoup(x, "html.parser").get_text())
    """if 'total_valid_imp' in df.columns:
        if df['total_valid_imp'].values[0] == 0:
            df['total_valid_imp'].replace(0, 'valeur inconnu')"""

    descriptions = [title]
    for index, row in df.iterrows():
        description = (
            f"De {row['begin']} à {row['end']}, l'événement avec l'ID {row['id']} "
            f"(dernière mise à jour le {row['lastUpdate']}) est causé par {row['cause']}. "
            f"La sévérité est {row.get('severity_text', 'non spécifiée')}. "
            f"Les balises associées sont {', '.join(row['tags'])}. "
            f"Le titre de l'événement est : {row['title']}. "
            f"Message : {row['message']}. "
            f"Message court : {row['shortMessage']}. "
            f"Points d'arrêt affectés : {', '.join(row['stop_points'])}. "
            f"Nom : {row['name']}. Mode : {row['mode']}. "
        )
        if 'status' in row:
            description += f"Statut : {row['status']}. "
        if 'priority' in row:
            description += f"Priorité : {row['priority']}. "
        if 'effect' in row:
            description += f"Effet : {row['effect']}. "
        if 'text' in row:
            description += f"Texte : {row['text']}. "
        if 'total_valid_imp' in row:
            description += f"Voyageurs impactés : {'valeur inconnu' if row['total_valid_imp'] == 0 else row['total_valid_imp']}."        
        descriptions.append(description)
    return descriptions

# Traiter chaque CSV séparément avec des titres distinctifs
file_paths = ['df_current_final.csv', 'df_preview_final.csv']
titles = ["Perturbations en cours :", "Perturbations à venir :"]
descriptions_list = [clean_and_describe_csv(file_path, title) for file_path, title in zip(file_paths, titles)]

# Réunir les descriptions des deux CSV dans une seule variable
combined_descriptions = "\n\n".join(["\n".join(descriptions) for descriptions in descriptions_list])

# Afficher les descriptions combinées
print("Descriptions combinées des deux CSV :\n")
print(combined_descriptions)

Descriptions combinées des deux CSV :

Perturbations en cours :
De 2025-03-27 18:42:00 à 2025-04-22 04:30:00, l'événement avec l'ID 26847bb8-0b33-11f0-b7af-0a58a9feac02 (dernière mise à jour le 20250327T184436) est causé par TRAVAUX. La sévérité est bloquante. Les balises associées sont Actualité. Le titre de l'événement est : Tramway T1 : Travaux - Trafic interrompu. Message : Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadrille Normandie-Niémen en raison de travaux. .Bus de remplacement.Plus d'informations sur le site ratp.fr. Message court : Trafic interrompu. Points d'arrêt affectés : Auguste Delaune, Bobigny - Pablo Picasso, Escadrille Normandie-Niemen, Gare de Noisy-le-Sec, Hôtel de Ville de Bobigny, Jean Rostand, La Ferme, Libération, Petit Noisy, Pont de Bondy. Nom : T1. Mode : Tramway. Statut : now. Priorité : 0.0. Effet : NO_SERVICE. Texte : Jusqu'au lundi 21 avril 2025, le trafic est interrompu entre Gare de Noisy-le-Sec et Escadril

Comment lire les indications temporelles

begin 20241012T020900

2024 : Année

10 : Mois (octobre)

12 : Jour

T : Séparateur entre la date et l'heure (utilisé pour indiquer que ce qui suit est l'heure)

02 : Heure (2 heures du matin)

09 : Minute

00 : Seconde